# TerraMind on SmallMinesDS — bi-temporal mine mapping with a frozen foundation model

**Question.** How well do the pretrained features of TerraMind (IBM/ESA, v1) map artisanal
gold-mining in south-western Ghana when the encoder is *not* tuned at all?

**Setup.**
- Input: for each 128 × 128 tile, the 2016 **and** 2022 patches — 13 bands each, split into
  TerraMind's native modalities `S2L2A` (10 optical bands), `S1RTC` (VV/VH) and `DEM`.
- Target: one 4-class map per pixel — `no mine` / `mine 2016 only` / `mine 2022 only (new)` /
  `mine both years`. The classes are mutually exclusive, so a standard segmentation head
  predicts both years' masks *and* the change map in one shot.
- Model: `terramind_v1_base` encoder, **frozen**, shared across the two years
  (`TemporalWrapper`, concat pooling) → UNet decoder trained from scratch.
- Ablation: S2 only → S2 + S1 → S2 + S1 + DEM, to see what the extra modalities buy.

**Compute.** Built for a Colab / Kaggle T4 (16 GB). The frozen-encoder run is ~1 min/epoch on a
T4. The whole thing also runs locally on CPU with `terramind_v1_tiny` and `SMOKE_TEST = True`
for pipeline checks.

Dataset: Ofori-Ampofo et al., *SmallMinesDS*, IEEE GRSL 2025 (`ellaampy/SmallMinesDS`).
Model: Jakubik et al., *TerraMind*, 2025 (`ibm-esa-geospatial/TerraMind-1.0-base`).

## 0. Runtime setup

Set `SMOKE_TEST = True` to run every cell end-to-end on a handful of tiles with the tiny model
(a few minutes on CPU). Set it to `False` on a GPU for the real run.

In [ ]:
SMOKE_TEST = False          # True: tiny model, 16 tiles, 1 epoch — just proves the pipeline
BACKBONE = "terramind_v1_tiny" if SMOKE_TEST else "terramind_v1_base"
EPOCHS = 1 if SMOKE_TEST else 30
BATCH_SIZE = 2 if SMOKE_TEST else 8
RUN_ABLATION = not SMOKE_TEST
SEED = 42

In [ ]:
import importlib.util, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if importlib.util.find_spec("terratorch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "terratorch>=1.2.13", "setuptools<81"])

In [ ]:
import json, os, time, warnings, zipfile
from pathlib import Path

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

import terratorch  # noqa: F401  (registers backbones)
from terratorch.tasks import SemanticSegmentationTask

warnings.filterwarnings("ignore", category=UserWarning)
L.seed_everything(SEED, workers=True)
plt.rcParams["figure.dpi"] = 110

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from importlib.metadata import version
print(f"torch {torch.__version__} | terratorch {version('terratorch')} | device {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

## 1. Data

On Colab the dataset is pulled straight from Hugging Face (1.65 GB zip) rather than uploaded
through Drive. Locally the notebook finds the already-extracted `Dataset/` and `data_splits/`.

In [ ]:
def acquire_dataset(root=Path(".")):
    try:
        from smallmines_data import find_data_root  # noqa
    except ImportError:
        pass  # module is written by the next cell; use the same search here
    for cand in (root / "Dataset", root / "SmallMinesDS" / "SmallMinesDS"):
        if (cand / "2016" / "IMAGE").is_dir():
            return cand
    from huggingface_hub import snapshot_download
    hf_dir = Path(snapshot_download(repo_id="ellaampy/SmallMinesDS", repo_type="dataset",
                                    local_dir=root / "SmallMinesDS"))
    with zipfile.ZipFile(hf_dir / "SmallMinesDS.zip") as z:
        z.extractall(hf_dir)
    return hf_dir / "SmallMinesDS"


DATA_ROOT = acquire_dataset()
SPLITS_DIR = next(p for p in (Path("data_splits"), Path("SmallMinesDS/data_splits"))
                  if (p / "train_test_splits_2016.csv").is_file())
print("images:", DATA_ROOT.resolve())
print("splits:", SPLITS_DIR.resolve())

### 1.1 Data pipeline module

The dataset/datamodule code lives in `smallmines_data.py` so the YAML config in
`configs/GFMS/terramind.yaml` can reference it. The cell below writes that file so the
notebook is self-contained on Colab (`%%writefile` is a no-op locally if the file is unchanged).

What it does:
- reads a tile's 2016 and 2022 GeoTIFFs and splits the 13 bands into `S2L2A` / `S1RTC` / `DEM`,
- standardises each band with **TerraMind's own pretraining mean/std** — the encoder is frozen,
  so inputs must look like what it was trained on,
- stacks the two years on a time axis → every modality is `(C, T=2, H, W)`,
- encodes the two binary masks as the 4-class target, mapping nodata (`-9999` / `255`) to
  zero after standardisation / `ignore_index`.

In [ ]:
%%writefile smallmines_data.py
"""SmallMinesDS -> TerraMind bi-temporal data pipeline.

Each tile has a 2016 and a 2022 patch (13 x 128 x 128) and a binary mine mask per year.
This module

- maps the 13 bands onto TerraMind's pretraining modalities (S2L2A, S1RTC, DEM),
- standardises them with TerraMind's own pretraining statistics,
- stacks the two years on a time axis so every modality is (C, T=2, H, W), which is what
  TerraTorch's ``TemporalWrapper`` expects,
- encodes the pair of binary masks as one 4-class target so per-year masks and the
  change map fall out of a standard semantic-segmentation head.

This file is also written out from ``terramind.ipynb`` (``%%writefile``) so the notebook
stays self-contained on Colab; keep the two in sync by editing the notebook cell.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import torch
from lightning import LightningDataModule
from torch.utils.data import DataLoader, Dataset

YEARS = ("2016", "2022")
IMAGE_NODATA = -9999.0
MASK_NODATA = 255
IGNORE_INDEX = -1

# Band layout of the 13-band SmallMinesDS GeoTIFF, expressed in TerraMind's band vocabulary.
# The dataset card lists the radar as Sentinel-1 RTC (terrain corrected + speckle filtered),
# which is its own TerraMind modality.
MODALITY_BANDS = {
    "S2L2A": ["BLUE", "GREEN", "RED", "RED_EDGE_1", "RED_EDGE_2", "RED_EDGE_3",
              "NIR_BROAD", "NIR_NARROW", "SWIR_1", "SWIR_2"],
    "S1RTC": ["VV", "VH"],
    "DEM": ["DEM"],
}
MODALITY_INDICES = {
    "S2L2A": list(range(0, 10)),
    "S1RTC": [10, 11],
    "DEM": [12],
}
ALL_MODALITIES = tuple(MODALITY_BANDS)

# TerraMind v1 pretraining statistics (from IBM/terramind configs). S2L2A is the 12-band
# list; the two bands this dataset lacks (COASTAL_AEROSOL, WATER_VAPOR) are dropped below.
_TM_S2L2A_BANDS = ["COASTAL_AEROSOL", "BLUE", "GREEN", "RED", "RED_EDGE_1", "RED_EDGE_2",
                   "RED_EDGE_3", "NIR_BROAD", "NIR_NARROW", "WATER_VAPOR", "SWIR_1", "SWIR_2"]
_TM_MEAN = {
    "S2L2A": [1390.458, 1503.317, 1718.197, 1853.910, 2199.100, 2779.975,
              2987.011, 3083.234, 3132.220, 3162.988, 2424.884, 1857.648],
    "S1RTC": [-10.93, -17.329],
    "DEM": [670.665],
}
_TM_STD = {
    "S2L2A": [2106.761, 2141.107, 2038.973, 2134.138, 2085.321, 1889.926,
              1820.257, 1871.918, 1753.829, 1797.379, 1434.261, 1334.311],
    "S1RTC": [4.391, 4.459],
    "DEM": [951.272],
}


def _subset(stats, modality):
    if modality == "S2L2A":
        return [stats[_TM_S2L2A_BANDS.index(b)] for b in MODALITY_BANDS["S2L2A"]]
    return stats


TERRAMIND_MEAN = {m: np.array(_subset(_TM_MEAN[m], m), dtype=np.float32) for m in ALL_MODALITIES}
TERRAMIND_STD = {m: np.array(_subset(_TM_STD[m], m), dtype=np.float32) for m in ALL_MODALITIES}

# 4-class target: mutually exclusive, so a standard softmax head predicts both years at once.
CLASS_NAMES = ["no mine", "mine 2016 only", "mine 2022 only (new)", "mine both years"]
NUM_CLASSES = len(CLASS_NAMES)


# --------------------------------------------------------------------------- paths & splits

def find_data_root(start=Path(".")):
    """Locate the folder holding <year>/IMAGE and <year>/MASK.

    Local layout is ``Dataset/2016/...``; the unzipped HF download is
    ``SmallMinesDS/SmallMinesDS/2016/...``.
    """
    start = Path(start)
    for cand in (start / "Dataset", start / "SmallMinesDS" / "SmallMinesDS", start / "SmallMinesDS"):
        if all((cand / y / "IMAGE").is_dir() and (cand / y / "MASK").is_dir() for y in YEARS):
            return cand
    raise FileNotFoundError(f"no <year>/IMAGE folders found under {start.resolve()}")


def find_splits_dir(start=Path(".")):
    start = Path(start)
    for cand in (start / "data_splits", start / "SmallMinesDS" / "data_splits"):
        if (cand / "train_test_splits_2016.csv").is_file():
            return cand
    raise FileNotFoundError(f"no data_splits/ folder found under {start.resolve()}")


def patch_paths(data_root, year, tile):
    data_root = Path(data_root)
    return (data_root / year / "IMAGE" / f"IMG_GH_{tile}_{year}.tif",
            data_root / year / "MASK" / f"MASK_GH_{tile}_{year}.tif")


def make_paired_split(splits_dir, out_csv, seed=42, val_frac=0.15, test_frac=0.15):
    """Build one tile-level split for the paired (2016, 2022) task.

    The published per-year CSVs were stratified independently, so 42% of tiles land in
    different splits in the two years. A model that sees both years of a tile needs a
    single assignment per tile, so we draw a fresh split stratified on the 2016->2022
    change in mine coverage (the quantity a change model has to get right).
    """
    from sklearn.model_selection import train_test_split

    frames = []
    for year in YEARS:
        df = pd.read_csv(Path(splits_dir) / f"train_test_splits_{year}.csv")
        df["tile"] = df["patch_name"].str.extract(r"_(\d{4})_")
        frames.append(df.set_index("tile")[["class_percentage", "split"]]
                        .rename(columns={"class_percentage": f"pct_{year}", "split": f"split_{year}"}))
    paired = frames[0].join(frames[1], how="inner").reset_index()
    paired["delta"] = paired["pct_2022"] - paired["pct_2016"]
    paired["change_bin"] = pd.cut(paired["delta"], bins=[-np.inf, 0, 1, 5, 15, np.inf],
                                  labels=["<=0", "0-1", "1-5", "5-15", ">15"])

    train_val, test = train_test_split(paired, test_size=test_frac, random_state=seed,
                                       stratify=paired["change_bin"])
    train, val = train_test_split(train_val, test_size=val_frac / (1 - test_frac),
                                  random_state=seed, stratify=train_val["change_bin"])
    paired["split"] = "train"
    paired.loc[paired.tile.isin(val.tile), "split"] = "val"
    paired.loc[paired.tile.isin(test.tile), "split"] = "test"
    paired = paired.sort_values("tile").reset_index(drop=True)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    paired.to_csv(out_csv, index=False)
    return paired


# --------------------------------------------------------------------------- dataset

def encode_target(mask_2016, mask_2022):
    """Two binary masks -> one 4-class map; nodata in either year becomes IGNORE_INDEX."""
    invalid = (mask_2016 == MASK_NODATA) | (mask_2022 == MASK_NODATA)
    target = (mask_2016 == 1).astype(np.int64) + 2 * (mask_2022 == 1).astype(np.int64)
    target[invalid] = IGNORE_INDEX
    return target


def decode_target(target):
    """Inverse of encode_target: (mine_2016, mine_2022, new_mining) boolean maps."""
    mine_2016 = (target == 1) | (target == 3)
    mine_2022 = (target == 2) | (target == 3)
    return mine_2016, mine_2022, target == 2


class SmallMinesPairedDataset(Dataset):
    """One item = one tile with both years stacked on a time axis.

    Returns ``{"image": {modality: float tensor (C, 2, H, W)}, "mask": long tensor (H, W),
    "filename": tile}``. The ``filename`` key is the one TerraTorch tasks already ignore.
    """

    def __init__(self, data_root, tiles, modalities=ALL_MODALITIES, augment=False):
        self.data_root = Path(data_root)
        self.tiles = list(tiles)
        self.modalities = list(modalities)
        self.augment = augment

    def __len__(self):
        return len(self.tiles)

    def _read(self, year, tile):
        img_path, mask_path = patch_paths(self.data_root, year, tile)
        with rasterio.open(img_path) as src:
            image = src.read().astype(np.float32)
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
        return image, mask

    def __getitem__(self, i):
        tile = self.tiles[i]
        images, masks = zip(*(self._read(year, tile) for year in YEARS))
        target = encode_target(*masks)

        sample = {}
        for mod in self.modalities:
            idx = MODALITY_INDICES[mod]
            x = np.stack([img[idx] for img in images], axis=1)  # (C, T, H, W)
            nodata = x == IMAGE_NODATA
            x = (x - TERRAMIND_MEAN[mod][:, None, None, None]) / TERRAMIND_STD[mod][:, None, None, None]
            x[nodata] = 0.0  # nodata sits at the pretraining mean after standardisation
            sample[mod] = x

        if self.augment:
            sample, target = self._d4(sample, target)

        return {
            "image": {mod: torch.from_numpy(np.ascontiguousarray(x)) for mod, x in sample.items()},
            "mask": torch.from_numpy(np.ascontiguousarray(target)),
            "filename": tile,
        }

    @staticmethod
    def _d4(sample, target):
        """Random dihedral flip/rotation applied identically to every modality and the target."""
        k = np.random.randint(4)
        flip = np.random.rand() < 0.5
        for mod, x in sample.items():
            x = np.rot90(x, k, axes=(-2, -1))
            sample[mod] = x[..., ::-1] if flip else x
        target = np.rot90(target, k, axes=(-2, -1))
        return sample, (target[..., ::-1] if flip else target)


class SmallMinesPairedDataModule(LightningDataModule):
    def __init__(self, data_root, split_csv, modalities=ALL_MODALITIES, batch_size=8,
                 num_workers=2, augment=True, limit=None):
        super().__init__()
        self.data_root = Path(data_root)
        self.split_csv = Path(split_csv)
        self.modalities = list(modalities)
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.augment = augment
        self.limit = limit  # cap tiles per split for smoke tests
        self.tiles = {}

    def setup(self, stage=None):
        df = pd.read_csv(self.split_csv, dtype={"tile": str})
        for split in ("train", "val", "test"):
            tiles = df.loc[df.split == split, "tile"].tolist()
            self.tiles[split] = tiles[: self.limit] if self.limit else tiles

    def _loader(self, split, shuffle):
        ds = SmallMinesPairedDataset(self.data_root, self.tiles[split], self.modalities,
                                     augment=self.augment and split == "train")
        return DataLoader(ds, batch_size=self.batch_size, shuffle=shuffle,
                          num_workers=self.num_workers, pin_memory=True,
                          persistent_workers=self.num_workers > 0, drop_last=shuffle)

    def train_dataloader(self):
        return self._loader("train", shuffle=True)

    def val_dataloader(self):
        return self._loader("val", shuffle=False)

    def test_dataloader(self):
        return self._loader("test", shuffle=False)

    def predict_dataloader(self):
        return self.test_dataloader()

In [ ]:
import importlib, smallmines_data
importlib.reload(smallmines_data)
from smallmines_data import (ALL_MODALITIES, CLASS_NAMES, IGNORE_INDEX, MODALITY_BANDS,
                             NUM_CLASSES, SmallMinesPairedDataModule, SmallMinesPairedDataset,
                             decode_target, make_paired_split, patch_paths)

### 1.2 One split per tile

The published `train_test_splits_<year>.csv` files were stratified **independently per year**,
so only 58 % of tiles land in the same split in both years. A model that sees both years of a
tile at once needs one assignment per tile, otherwise a tile's 2016 image is trained on and its
2022 image is "tested" on — the same ground. We draw a fresh, seeded 70/15/15 split stratified
on the 2016→2022 change in mine coverage and save it to `data_splits/paired_splits.csv`.

In [ ]:
PAIRED_CSV = SPLITS_DIR / "paired_splits.csv"
paired = make_paired_split(SPLITS_DIR, PAIRED_CSV, seed=SEED)

print(f"{len(paired)} tile pairs -> {PAIRED_CSV}")
display(pd.crosstab(paired["split"], paired["change_bin"], margins=True))
print(f"\nmean mine coverage  2016 {paired.pct_2016.mean():.2f}%   2022 {paired.pct_2022.mean():.2f}%")
print(f"tiles with any growth: {(paired.delta > 0).mean():.0%}, >5 pp growth: {(paired.delta > 5).mean():.0%}")

In [ ]:
datamodule = SmallMinesPairedDataModule(
    DATA_ROOT, PAIRED_CSV, modalities=ALL_MODALITIES, batch_size=BATCH_SIZE,
    num_workers=0 if SMOKE_TEST else 2, limit=16 if SMOKE_TEST else None,
)
datamodule.setup()
print({k: len(v) for k, v in datamodule.tiles.items()})

batch = next(iter(datamodule.val_dataloader()))
for mod, x in batch["image"].items():
    print(f"{mod:6s} {tuple(x.shape)}  mean {x.mean():+.2f}  std {x.std():.2f}")
print("mask  ", tuple(batch["mask"].shape), batch["mask"].dtype)

counts = torch.bincount(batch["mask"][batch["mask"] >= 0].flatten(), minlength=NUM_CLASSES)
print("class pixel share in this batch:",
      {n: f"{c / counts.sum():.1%}" for n, c in zip(CLASS_NAMES, counts)})

Sanity check: true-colour composites for both years and the 4-class target of one val tile.

In [ ]:
def stretch(band, low=2, high=98):
    band = np.asarray(band, dtype=np.float32)
    valid = band[np.isfinite(band) & (band != -9999)]
    if valid.size == 0:
        return np.zeros_like(band)
    lo, hi = np.percentile(valid, [low, high])
    return np.clip((band - lo) / max(hi - lo, 1e-6), 0, 1)


def rgb(image):
    return np.dstack([stretch(image[b]) for b in (2, 1, 0)])


from matplotlib.colors import ListedColormap
CLASS_CMAP = ListedColormap(["#f2f2f2", "#4c72b0", "#dd3c3c", "#7a1f1f"])


def show_tile(tile, pred=None, ds=None):
    ds = ds or SmallMinesPairedDataset(DATA_ROOT, [tile])
    (img16, m16), (img22, m22) = ds._read("2016", tile), ds._read("2022", tile)
    target = smallmines_data.encode_target(m16, m22)
    panels = [("2016", rgb(img16)), ("2022", rgb(img22)), ("target", target)]
    if pred is not None:
        panels.append(("prediction", pred))
    fig, axes = plt.subplots(1, len(panels), figsize=(3.4 * len(panels), 3.6))
    for ax, (title, arr) in zip(axes, panels):
        if arr.ndim == 2:
            ax.imshow(np.ma.masked_where(arr < 0, arr), cmap=CLASS_CMAP, vmin=0, vmax=3,
                      interpolation="nearest")
        else:
            ax.imshow(arr)
        ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"tile {tile}   (grey none | blue 2016-only | red new 2022 | dark-red both)",
                 fontsize=9)
    plt.tight_layout()


show_tile(paired.loc[paired.split == "val"].sort_values("delta").tile.iloc[-1])

## 2. Model

`EncoderDecoderFactory` assembles: TerraMind encoder (pretrained weights from Hugging Face,
band subset selected by name so the 10 available S2 bands pick up their pretrained patch-embed
weights) → `TemporalWrapper` running the encoder on 2016 and 2022 with shared weights and
concatenating the token features → necks that reshape 4 transformer layers into a feature
pyramid → `UNetDecoder` → 4-class head.

`freeze_backbone=True` means **only the decoder and head train**; TerraMind itself stays as
released. Dice loss is used because ~90 % of pixels are `no mine`.

In [ ]:
LAYER_INDICES = {  # which transformer blocks feed the pyramid (4 of depth)
    "terramind_v1_tiny": [2, 5, 8, 11], "terramind_v1_small": [2, 5, 8, 11],
    "terramind_v1_base": [2, 5, 8, 11], "terramind_v1_large": [5, 11, 17, 23],
}


def build_task(modalities, backbone=BACKBONE, lr=1e-4, freeze_backbone=True):
    model_args = dict(
        backbone=backbone,
        backbone_pretrained=True,
        backbone_modalities=list(modalities),
        backbone_bands={m: MODALITY_BANDS[m] for m in modalities},
        backbone_merge_method="mean",          # average the per-modality tokens
        backbone_use_temporal=True,            # shared encoder over the 2 years ...
        backbone_temporal_n_timestamps=2,
        backbone_temporal_pooling="concat",    # ... then concat 2016 | 2022 features
        necks=[
            {"name": "SelectIndices", "indices": LAYER_INDICES[backbone]},
            {"name": "ReshapeTokensToImage", "remove_cls_token": False},
            {"name": "LearnedInterpolateToPyramidal"},
        ],
        decoder="UNetDecoder",
        decoder_channels=[512, 256, 128, 64],
        head_dropout=0.1,
        num_classes=NUM_CLASSES,
    )
    return SemanticSegmentationTask(
        model_args=model_args,
        model_factory="EncoderDecoderFactory",
        loss="dice",
        ignore_index=IGNORE_INDEX,
        lr=lr,
        optimizer="AdamW",
        optimizer_hparams={"weight_decay": 0.05},
        scheduler="ReduceLROnPlateau",
        scheduler_hparams={"factor": 0.5, "patience": 3},  # monitors val/loss
        freeze_backbone=freeze_backbone,
        class_names=CLASS_NAMES,
        plot_on_val=False,
    )


def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


task = build_task(ALL_MODALITIES)
total, trainable = count_params(task.model)
enc_total, enc_trainable = count_params(task.model.encoder)
print(f"{BACKBONE}: {enc_total/1e6:.1f} M encoder params ({enc_trainable/1e6:.1f} M trainable)")
print(f"whole model: {total/1e6:.1f} M params, {trainable/1e6:.1f} M trainable (decoder + head)")

In [ ]:
# Dry run: one batch through the untrained model, to confirm shapes before spending GPU time.
task.eval()
with torch.no_grad():
    out = task.model({m: x for m, x in batch["image"].items()})
print("logits", tuple(out.output.shape), "  target", tuple(batch["mask"].shape))
assert out.output.shape[-2:] == batch["mask"].shape[-2:]

## 3. Training

Checkpoint selection and early stopping use validation mIoU (`val/mIoU`).
Logs, checkpoints and metrics go under `output/<run name>/`. On Colab, either mount Drive and
point `OUT_DIR` there, or download `output/` before the runtime dies.

In [ ]:
OUT_DIR = Path("output")


def run_experiment(name, modalities, epochs=EPOCHS, backbone=BACKBONE, lr=1e-4):
    L.seed_everything(SEED, workers=True)
    dm = SmallMinesPairedDataModule(
        DATA_ROOT, PAIRED_CSV, modalities=modalities, batch_size=BATCH_SIZE,
        num_workers=0 if SMOKE_TEST else 2, limit=16 if SMOKE_TEST else None,
    )
    task = build_task(modalities, backbone=backbone, lr=lr)
    logger = CSVLogger(OUT_DIR, name=name, version="")
    ckpt = ModelCheckpoint(monitor="val/mIoU", mode="max",
                           save_top_k=1, filename="best", save_last=True)
    trainer = L.Trainer(
        accelerator="auto", devices=1, max_epochs=epochs,
        precision="16-mixed" if DEVICE == "cuda" else "32-true",
        logger=logger, log_every_n_steps=10, enable_progress_bar=True,
        callbacks=[ckpt, LearningRateMonitor(logging_interval="epoch"),
                   EarlyStopping(monitor="val/mIoU", mode="max", patience=8)],
    )
    t0 = time.time()
    trainer.fit(task, datamodule=dm)
    print(f"[{name}] trained {trainer.current_epoch} epochs in {(time.time() - t0) / 60:.1f} min,"
          f" best val mIoU {ckpt.best_model_score:.4f}")
    test_metrics = trainer.test(task, datamodule=dm, ckpt_path=ckpt.best_model_path or None, verbose=False)[0]
    return dict(name=name, task=task, datamodule=dm, trainer=trainer,
                ckpt_path=ckpt.best_model_path, test_metrics=test_metrics)


MAIN_RUN = "terramind_frozen_S2_S1_DEM"
main = run_experiment(MAIN_RUN, ALL_MODALITIES)

In [ ]:
def plot_curves(name):
    log = pd.read_csv(OUT_DIR / name / "metrics.csv")
    ep = log.groupby("epoch").mean(numeric_only=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for col, lab in (("train/loss", "train"), ("val/loss", "val")):
        if col in ep: axes[0].plot(ep.index, ep[col], label=lab)
    axes[0].set_title("dice loss"); axes[0].legend()
    if "val/mIoU" in ep:
        axes[1].plot(ep.index, ep["val/mIoU"])
    axes[1].set_title("val mIoU (4 classes)")
    for ax in axes: ax.set_xlabel("epoch"); ax.grid(alpha=.3)
    fig.suptitle(name); plt.tight_layout()


plot_curves(MAIN_RUN)

## 4. Evaluation on the held-out test tiles

TerraTorch reports 4-class mIoU. For the research question we also want the things a
mining-monitoring user cares about, decoded from the 4-class prediction:

- **mine 2016** and **mine 2022** — per-year mine extent (IoU / F1 / precision / recall),
- **new mining 2016→2022** — the change map (class `mine 2022 only`),
- the 4 × 4 confusion matrix, to see which confusions dominate.

In [ ]:
def collect_predictions(task, loader):
    task.eval().to(DEVICE)
    preds, targets, tiles = [], [], []
    with torch.no_grad():
        for b in loader:
            x = {m: t.to(DEVICE) for m, t in b["image"].items()}
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                logits = task.model(x).output
            preds.append(logits.argmax(1).cpu()); targets.append(b["mask"]); tiles += list(b["filename"])
    return torch.cat(preds).numpy(), torch.cat(targets).numpy(), tiles


def binary_scores(pred, true, valid):
    p, t = pred[valid], true[valid]
    tp, fp, fn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum()
    iou = tp / max(tp + fp + fn, 1); prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(IoU=iou, F1=2 * prec * rec / max(prec + rec, 1e-9), precision=prec, recall=rec,
                positive_share=t.mean())


def evaluate(result):
    task, dm = result["task"], result["datamodule"]
    if result["ckpt_path"]:
        task.load_state_dict(torch.load(result["ckpt_path"], map_location="cpu")["state_dict"])
    pred, true, tiles = collect_predictions(task, dm.test_dataloader())
    valid = true != IGNORE_INDEX
    rows = {}
    for label, (pm, tm) in zip(("mine 2016", "mine 2022", "new mining 2016->2022"),
                               zip(decode_target(pred), decode_target(true))):
        rows[label] = binary_scores(pm, tm, valid)
    scores = pd.DataFrame(rows).T
    cm = pd.crosstab(pd.Series(true[valid], name="true").map(dict(enumerate(CLASS_NAMES))),
                     pd.Series(pred[valid], name="pred").map(dict(enumerate(CLASS_NAMES))),
                     normalize="index")
    return dict(scores=scores, confusion=cm, pred=pred, true=true, tiles=tiles,
                miou=result["test_metrics"].get("test/mIoU", np.nan))


main_eval = evaluate(main)
print(f"4-class test mIoU (TerraTorch): {main_eval['miou']:.4f}\n")
display(main_eval["scores"].style.format("{:.3f}"))
display(main_eval["confusion"].style.format("{:.2%}").background_gradient(cmap="Blues", axis=None))

In [ ]:
# Predictions on the test tiles with the largest true mine growth.
test_meta = paired.set_index("tile").loc[main_eval["tiles"]]
order = np.argsort(-test_meta["delta"].values)[:4]
for i in order:
    show_tile(main_eval["tiles"][i], pred=main_eval["pred"][i])

## 5. Modality ablation

Same recipe, fewer inputs. If radar and elevation carry information the optical bands do not
(bare-earth pits under haze, terrain context for river-bed mining), the full model should win on
the `new mining` row in particular.

In [ ]:
ABLATION = {
    "terramind_frozen_S2":        ["S2L2A"],
    "terramind_frozen_S2_S1":     ["S2L2A", "S1RTC"],
    MAIN_RUN:                     list(ALL_MODALITIES),
}
results = {MAIN_RUN: (main, main_eval)}

if RUN_ABLATION:
    for name, mods in ABLATION.items():
        if name not in results:
            r = run_experiment(name, mods)
            results[name] = (r, evaluate(r))

table = pd.DataFrame({
    name: {"modalities": "+".join(ABLATION[name]).replace("L2A", "").replace("RTC", ""),
           "4-class mIoU": ev["miou"],
           **{f"{row} IoU": ev["scores"].loc[row, "IoU"] for row in ev["scores"].index},
           "new mining F1": ev["scores"].loc["new mining 2016->2022", "F1"]}
    for name, (r, ev) in results.items()
}).T
display(table.style.format(precision=3))
OUT_DIR.mkdir(exist_ok=True)
table.to_csv(OUT_DIR / "ablation_results.csv")
for name, (r, ev) in results.items():
    ev["scores"].to_csv(OUT_DIR / name / "test_scores.csv")
    ev["confusion"].to_csv(OUT_DIR / name / "test_confusion.csv")
print("saved to", OUT_DIR.resolve())

## 6. Notes for the write-up

- **What was tested.** TerraMind's *pretrained representation* — the encoder was never updated.
  Every number above is "how linearly-separable-ish is mining in TerraMind feature space", not
  "how good can a TerraMind-based model get". A full fine-tune (`freeze_backbone=False`,
  `lr=2e-5`) is the natural next experiment and is one flag away.
- **Why 4 classes.** Predicting both years as a joint label keeps the task a single-head
  segmentation problem while still yielding a change map; `mine 2016 only` is rare and is the
  weakest class — treat its scores with caution (report support alongside).
- **Why a new split.** The published per-year splits disagree for 42 % of tiles; the paired
  split is stratified on change magnitude and seeded (`data_splits/paired_splits.csv`). Numbers
  are therefore *not* directly comparable to the per-year benchmarks in the dataset paper.
- **Normalisation.** Inputs are standardised with TerraMind's pretraining statistics, not the
  dataset's; with a frozen encoder this matters more than usual.
- **Radar offset.** After standardisation with TerraMind's S1RTC statistics this dataset's VV/VH
  sit ~1-2 σ above zero (its backscatter is brighter than TerraMesh's average). The frozen
  encoder still sees plausible values, but if the S1 ablation shows no gain, re-standardising
  radar with dataset statistics is the first thing to try.
- **Missing bands.** Only 10 of TerraMind's 12 S2L2A bands exist here; the patch-embedding
  weights for the 10 present bands are reused, the two missing ones are simply absent.
- **Resolution.** 128 × 128 at 10 m with 16-px patches gives an 8 × 8 token grid per year;
  the decoder has to recover 16× spatial detail. Bilinear-upsampling inputs to 224 × 224 is an
  easy variant to try if boundaries look coarse.